In [122]:
import os
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.preprocessing import FunctionTransformer, LabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

In [123]:
# with open(model_path, 'rb') as model_file:
#     best_model = pickle.load(model_file)
# print(f"CV ROC-AUC: {best_model.best_score_:.4f}")
# predictions = best_model.predict_proba(test)[:, 1]
# submission = pd.DataFrame({'id': test_id, 'loan_paid_back': predictions})
# submission_path = os.path.join(MODEL_DIR, f'submission_{model_filename}.csv')
# submission.to_csv(f'submission{model_filename}.csv', index=False)BASE_DIR = Path.cwd().parent
BASE_DIR = Path.cwd().parent
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_FILE = os.path.join(DATA_DIR, "train_raw.csv")
TEST_FILE = os.path.join(DATA_DIR, "test_raw.csv")
MODEL_DIR = os.path.join(BASE_DIR, "models")
df = pd.read_csv(TRAIN_FILE, index_col=0)
df.head()

,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
id,,,,,,,,,,,,
0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [124]:
train = pd.read_csv(TRAIN_FILE, index_col=0)
test = pd.read_csv(TEST_FILE)
test_id = test['id']
test.drop(columns=['id'])

# Target
y_train = train['loan_paid_back']

In [125]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import TargetEncoder, KBinsDiscretizer
from itertools import combinations, product
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import lightgbm as lgb


def extract_all_digits(df):
    digit_features = {}
    numeric_cols = ['credit_score', 'loan_amount', 'interest_rate', 'annual_income', 'debt_to_income_ratio']

    for col in numeric_cols:
        col_data = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0)

        col_int = col_data.astype(int)
        first_digit = np.where(col_int > 0,
                               col_int // 10 ** np.floor(np.log10(np.maximum(col_int, 1))).astype(int), 0)
        digit_features[f'{col}_first_digit'] = first_digit.astype(int)

        digit_features[f'{col}_digit_sum'] = col_data.astype(str).apply(
            lambda x: sum(int(d) for d in str(x).replace('.', '') if d.isdigit())
        )

        # Last digit
        digit_features[f'{col}_last_digit'] = (col_data % 10).astype(int)

    return pd.DataFrame(digit_features)


def digit_interactions(df):
    digits = extract_all_digits(df)
    interactions = []

    for pair in combinations(digits.columns[:6], 2):
        col1, col2 = digits[pair[0]], digits[pair[1]]
        interactions.append((col1 * 10 + col2).rename(f'digit_{pair[0]}_{pair[1]}'))

    return pd.concat(interactions, axis=1)

def round_features(df):
    rounds = {}
    numeric_cols = ['loan_amount', 'annual_income', 'credit_score']

    for col in numeric_cols:
        rounds[f'{col}_round100'] = np.round(df[col] / 100) * 100
        rounds[f'{col}_floor100'] = np.floor(df[col] / 100) * 100

    return pd.DataFrame(rounds)


def base_features_te(df, y):
    """Target encode base features - FIXED"""
    base_cols = ['grade_subgrade', 'employment_status', 'loan_purpose']
    te = TargetEncoder(smooth=10.0)
    encoded = te.fit_transform(df[base_cols], y)
    return pd.DataFrame(encoded)  # BEZ columns= !

def employment_te(df):
    """TE base features using employment_status as target - FIXED"""
    employment_map = {'Unemployed': 0, 'Employed': 1, 'Retired': 2}
    emp_target = df['employment_status'].map(employment_map).fillna(1)
    base_cols = ['grade_subgrade', 'loan_purpose', 'education_level']

    te = TargetEncoder(smooth=5.0)
    encoded = te.fit_transform(df[base_cols], emp_target)

    # Użyj DEFAULT nazw kolumn (9 kolumn automatycznie)
    return pd.DataFrame(encoded)  # BEZ columns= !

# 6. Quantile bins + categorical credit_score
def misc_features(df):
    """Fix KBinsDiscretizer"""
    misc = {}
    kb = KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile',
                         quantile_method='averaged_inverted_cdf')  # Fix warning
    misc['credit_score_bin'] = kb.fit_transform(df[['credit_score']]).ravel().astype(int)
    misc['debt_bin'] = kb.fit_transform(df[['debt_to_income_ratio']]).ravel().astype(int)
    misc['credit_score_cat'] = (df['credit_score'] // 10).astype(int)
    return pd.DataFrame(misc)

# 7. COMBINE ALL
def ultimate_feature_engineering(df, y=None):
    features = pd.DataFrame()

    # Base
    features['debt_x_credit'] = df['debt_to_income_ratio'] * df['credit_score']
    features['debt_log'] = np.log1p(df['debt_to_income_ratio'].clip(lower=0))

    # Digits (numeric)
    digits = extract_all_digits(df)
    features = pd.concat([features, digits.iloc[:, :8]], axis=1)

    # Numeric digit interactions
    digit_int = digit_interactions(df).iloc[:, :10]
    features = pd.concat([features, digit_int], axis=1)

    # Rounds
    rounds = round_features(df)
    features = pd.concat([features, rounds], axis=1)

    # Misc
    misc = misc_features(df)
    features = pd.concat([features, misc], axis=1)

    if y is not None:
        base_te = base_features_te(df, y)
        emp_te = employment_te(df)
        features = pd.concat([features, base_te, emp_te], axis=1)

    # KLUCZOWY FIX - wszystko NUMERIC!
    return features.select_dtypes(include=[np.number]).fillna(0).astype('float32')


# 8. PIPELINE VERSION
X = ultimate_feature_engineering(train.drop(columns=['loan_paid_back']), y_train)
print("✅ All numeric:", (X.dtypes == 'float32').all())
print("Shape:", X.shape)
print("Sample dtypes:\n", X.dtypes[:5])

✅ All numeric: True
Shape: (593994, 41)
Sample dtypes:
 debt_x_credit               float32
debt_log                    float32
credit_score_first_digit    float32
credit_score_digit_sum      float32
credit_score_last_digit     float32
dtype: object


In [126]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold, cross_val_score
#
# N_JOBS = 2
#
# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("classifier", LogisticRegression(
#         max_iter=1000,
#         solver="liblinear",
#         C=1.0,
#     ))
# ])
#
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#
# scores = cross_val_score(
#     pipe,
#     train,
#     y_train,
#     cv=cv,
#     scoring="roc_auc",
#     n_jobs=N_JOBS
# )
# print("Baseline ROC-AUC:", scores.mean(), "+/-", scores.std())


In [127]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

N_JOBS = 1

# POPRAWIONY preprocessor (z poprzedniego kodu)
pipe = Pipeline([ # Twój działający preprocessor
    ("classifier", RandomForestClassifier())  # Placeholder
])

# POPRAWIONY search_space - DODANE num_leaves i poprawne parametry
search_space = [
    {
        "classifier": [LGBMClassifier(
            random_state=42,
            device="gpu",
            verbose=-1,  # Wyłącz verbose w GridSearch
            n_jobs=N_JOBS,
            max_bin=63
        )],
        "classifier__n_estimators": [200, 400],           # Więcej = lepiej
        "classifier__learning_rate": [0.03, 0.05],        # Wolniejsze uczenie = stabilne
        "classifier__max_depth": [8, 12, -1],             # Głębsze drzewa
        "classifier__min_child_samples": [100, 200],      # Regularizacja
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gridsearch = GridSearchCV(
    pipe,
    search_space,
    cv=cv,
    scoring='roc_auc',
    n_jobs=1,        # ← KLUCZOWE dla GPU
    verbose=2        # Mniej spamu
)
print("🔍 DUPLIKATY W X:")
duplicates = X.columns[X.columns.duplicated()].tolist()
print("Duplicate columns:", duplicates)
print("Total duplicates:", len(duplicates))

# SZYBKI FIX (bezpieczny)
if len(duplicates) > 0:
    print("🛠️ Usuwam duplikaty...")
    X = X.loc[:, ~X.columns.duplicated()].copy()
    print("✅ Po usunięciu:", X.shape)
best_model = gridsearch.fit(X, y_train)

🔍 DUPLIKATY W X:
Duplicate columns: [0, 1, 2]
Total duplicates: 3
🛠️ Usuwam duplikaty...
✅ Po usunięciu: (593994, 38)
Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV] END classifier=LGBMClassifier(device='gpu', max_bin=63, n_jobs=1, random_state=42, verbose=-1), classifier__learning_rate=0.03, classifier__max_depth=8, classifier__min_child_samples=100, classifier__n_estimators=200; total time=   4.2s
[CV] END classifier=LGBMClassifier(device='gpu', max_bin=63, n_jobs=1, random_state=42, verbose=-1), classifier__learning_rate=0.03, classifier__max_depth=8, classifier__min_child_samples=100, classifier__n_estimators=200; total time=   4.2s
[CV] END classifier=LGBMClassifier(device='gpu', max_bin=63, n_jobs=1, random_state=42, verbose=-1), classifier__learning_rate=0.03, classifier__max_depth=8, classifier__min_child_samples=100, classifier__n_estimators=200; total time=   4.3s
[CV] END classifier=LGBMClassifier(device='gpu', max_bin=63, n_jobs=1, random_state=42, verbos

In [128]:
import datetime

best_model_name = str(best_model.best_estimator_['classifier'].__class__.__name__)
best_model_score = f"{best_model.best_score_}"
model_filename = f'{best_model_name}{best_model_score}.pkl'
model_folder = os.path.join(MODEL_DIR, str(datetime.datetime.now()))
os.mkdir(model_folder)
model_path = os.path.join(model_folder, model_filename)
with open(model_path, "wb") as model_file:
    pickle.dump(best_model, model_file)

print(f"Najlepszy model: {best_model_name}")
print(f"CV ROC-AUC: {best_model.best_score_:.4f}")
print(f"Ranking CV: {best_model.cv_results_['mean_test_score'][-5:]}")

Najlepszy model: LGBMClassifier
CV ROC-AUC: 0.9163
Ranking CV: [0.9163431  0.91594052 0.91628961 0.91598618 0.91634564]


In [129]:
with open(model_path, 'rb') as model_file:
    best_model = pickle.load(model_file)
print(f"CV ROC-AUC: {best_model.best_score_:.4f}")
predictions = best_model.predict_proba(test)[:, 1]
submission = pd.DataFrame({'id': test_id, 'loan_paid_back': predictions})
submission_path = os.path.join(model_folder, f'submission_{model_filename}.csv')
submission.to_csv(submission_path, index=False)

CV ROC-AUC: 0.9163


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: gender: str, marital_status: str, education_level: str, employment_status: str, loan_purpose: str, grade_subgrade: str